### Purpose

이카운트 품목등록 엑셀(`docs/ESA009M.xlsx`)의 **기타 코드** — D(기타물품)·E(기타: 운반비·택배비 등)·G(잔디) — 중
약품 DB(`dschemicals`)에 없는 것을 넣는다. 일회성.

- 저장은 약품과 같은 표 (`dschemicals`). 웹은 "약품정보"(A·B·C)와 "기타 코드(잔디/물품)"(D·E·G) 탭으로 나눠 보여 준다.
  재고·견적은 지금처럼 `GET /dschemical`로 다 읽는다.
- 변환은 웹 [이카운트 비교]의 `ecountToChemical`과 같다: 구입가 = 입고단가, 판가·용역판가 0, 중요도 `신규구매`, `origin: ecount`.
  분류: D → 기타물품/기타물품, E → 기타/기타, G → 잔디/잔디.
- 사용 = YES 인 품목만. **이미 있는 코드는 건너뛴다** (조건부 저장, 덮어쓰지 않음).
- 셀 3(미리보기)은 쓰지 않는다. 셀 4가 쓴다.

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import boto3
import openpyxl
from botocore.exceptions import ClientError

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
table = boto3.resource('dynamodb').Table('dschemicals')

Settings

In [ ]:
EXCEL = Path('..', '..', 'docs', 'ESA009M.xlsx')
PREFIX_CLASS = {'D': ('기타물품', '기타물품'), 'E': ('기타', '기타'), 'G': ('잔디', '잔디')}   # 앞 글자 → (대분류, 중분류)
PROD_TYPES = {'원재료', '제품', '반제품', '상품', '부재료', '무형상품'}

def read_excel(path):
    ws = openpyxl.load_workbook(path, data_only=True).worksheets[0]
    rows = list(ws.iter_rows(values_only=True))
    hi = next(i for i, r in enumerate(rows) if r and '품목코드' in [str(c or '').strip() for c in r])
    header = [str(c or '').strip() for c in rows[hi]]
    return [dict(zip(header, r)) for r in rows[hi + 1:] if r and r[header.index('품목코드')]]

def to_item(row, now):
    code = str(row['품목코드']).strip()
    infoL2, infoL1 = PREFIX_CLASS[code[0]]
    kind = str(row.get('품목구분') or '').strip('[] ')
    return {
        'dsids': code, 'name': str(row['품목명']).strip(), 'unit': str(row.get('규격정보') or '').strip(),
        'infoL2': infoL2, 'infoL1': infoL1, 'infoL3': '신규구매',
        'IN_PRICE': Decimal(str(row.get('입고단가') or 0)), 'OUT_PRICE': Decimal(0), 'OUT_PRICE1': Decimal(0),
        'IN_PRICE[2025]': '', 'OUT_PRICE[2025]': '', 'OUT_PRICE1[2025]': '',
        'type': f'[{kind}]' if kind in PROD_TYPES else '', 'category': '기타', 'vendors': '', 'cost': '', 'history': '',
        'usage': Decimal(0), 'active': 'Y', 'flgWork': 'N', 'flgOut': 'N', 'origin': 'ecount', 'createdAt': now,
    }

3. 미리보기 (쓰지 않음)

In [ ]:
rows = read_excel(EXCEL)
targets = [r for r in rows if str(r['품목코드']).strip()[:1] in PREFIX_CLASS]
unused = [r for r in targets if str(r.get('사용') or '').strip() != 'YES']
targets = [r for r in targets if str(r.get('사용') or '').strip() == 'YES']
existing = set()
kw = {'ProjectionExpression': 'dsids'}
while True:
    page = table.scan(**kw)
    existing |= {i['dsids'] for i in page['Items']}
    if 'LastEvaluatedKey' not in page:
        break
    kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
now = datetime.now(timezone.utc).isoformat(timespec='milliseconds').replace('+00:00', 'Z')
plan = [to_item(r, now) for r in targets if str(r['품목코드']).strip() not in existing]
print(f"엑셀 기타 코드 {len(targets)}건 (사용 안 함 {len(unused)}건 제외) · 이미 있음 {len(targets) - len(plan)} · 넣을 것 {len(plan)}")
for p in 'DEG':
    part = [i for i in plan if i['dsids'].startswith(p)]
    print(f'  {p} {len(part)}건:', ', '.join(f"{i['dsids']} {i['name']}({i['unit']})" for i in part))

4. 넣기 — 코드가 없을 때만 (조건부)

In [ ]:
added, skipped = [], []
for item in plan:
    try:
        table.put_item(Item=item, ConditionExpression='attribute_not_exists(dsids)')
        added.append(item['dsids'])
    except ClientError as e:
        if e.response['Error']['Code'] != 'ConditionalCheckFailedException':
            raise
        skipped.append(item['dsids'])
print(f'넣음 {len(added)} · 그사이 이미 생김 {len(skipped)}', skipped or '')